<a href="https://colab.research.google.com/github/zahid1704/ML_Project/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zahid1704/ML_Project/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

one row in fact_content_daily_performance is one page (content_hash_id) for one client (client_hash_id) on one day (report_date). I will collapse March into one row per page per client, so each row summarizes that page’s signals over the month. March 2026 (month=2026-03), from 2026-03-01 to 2026-03-31. I chose a mid-panel month because the final month (June 2026) is the sealed test month, and the daily data starts in January 2025, so March has history behind it for most clients.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
MONTH = "2026-03"
START_DATE, END_DATE = "2026-03-01", "2026-03-31"
RAW_GRAIN = ["report_date", "client_hash_id", "content_hash_id"]
ANALYSIS_GRAIN = ["client_hash_id", "content_hash_id"]

print("Month:", MONTH, "|", START_DATE, "to", END_DATE)
print("Raw grain:", RAW_GRAIN)
print("Analysis grain (one row per page per client):", ANALYSIS_GRAIN)


Month: 2026-03 | 2026-03-01 to 2026-03-31
Raw grain: ['report_date', 'client_hash_id', 'content_hash_id']
Analysis grain (one row per page per client): ['client_hash_id', 'content_hash_id']


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Label (proxy): low_ctr_for_position, which is 1 when a page’s CTR is below the median CTR of pages in the same position tier. This is a proxy computed from observed CTR, not a future outcome.

Features: impressions (demand), avg_position (rank), word_count (content depth), content_age_days (age), and one engagement signal such as scroll_rate. All of these are observed measurements, available at the time of the daily record.

Context (not model inputs): client_hash_id (for grouping and client-held-out checks), content_hash_id (join key), report_date (time window), and ga4_data_available (shows whether analytics tracking existed).

Excluded, with reasons:

ctr and clicks: the label is built from CTR, so using either as a feature would leak the answer.
sessions_ai: AI-referral sessions are very sparse (about 30,000 of 79 million daily rows), so they would be mostly empty and misleading.
Any product score or flag (health_score, priority_score, action_type): not in my data, and using one would copy the product’s own decision.
Raw URLs, queries, titles, client names: removed from the release for privacy, and I will not try to reconstruct them.

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
fields = {
    "label": ["low_ctr_for_position"],
    "feature": ["impressions", "avg_position", "word_count", "content_age_days", "scroll_rate"],
    "context": ["client_hash_id", "content_hash_id", "report_date", "ga4_data_available"],
    "excluded": {
        "ctr": "label is built from it, would leak",
        "clicks": "label is built from CTR, would leak",
        "sessions_ai": "too sparse to be reliable",
    },
}
for bucket, cols in fields.items():
    print(bucket, "->", cols)


label -> ['low_ctr_for_position']
feature -> ['impressions', 'avg_position', 'word_count', 'content_age_days', 'scroll_rate']
context -> ['client_hash_id', 'content_hash_id', 'report_date', 'ga4_data_available']
excluded -> {'ctr': 'label is built from it, would leak', 'clicks': 'label is built from CTR, would leak', 'sessions_ai': 'too sparse to be reliable'}


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import duckdb
from google.colab import userdata

con = duckdb.connect()
token = userdata.get("HF_TOKEN")
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{token}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
print("Connected")

Connected


In [16]:
files = con.sql(f"SELECT * FROM glob('{REL}/fact_content_daily_performance/month=2026-03/*')").df()
print(files)

                                                file
0  hf://datasets/FlyRank/internship-warehouse/fac...


In [17]:
con.execute(f"""
CREATE OR REPLACE VIEW march AS
SELECT * FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')
""")
print(con.sql("DESCRIBE march").df()[["column_name", "column_type"]])

                 column_name column_type
0                report_date        DATE
1             client_hash_id     VARCHAR
2            content_hash_id     VARCHAR
3             client_has_gsc     BOOLEAN
4             client_has_ga4     BOOLEAN
5         gsc_data_available     BOOLEAN
6         ga4_data_available     BOOLEAN
7            gsc_impressions      BIGINT
8                 gsc_clicks      BIGINT
9           gsc_sum_position      BIGINT
10          gsc_avg_position      DOUBLE
11             ga4_pageviews      BIGINT
12              ga4_sessions      BIGINT
13                 ga4_users      BIGINT
14      ga4_engaged_sessions      BIGINT
15  ga4_total_engagement_sec      BIGINT
16          sessions_organic      BIGINT
17           sessions_direct      BIGINT
18         sessions_referral      BIGINT
19           sessions_social      BIGINT
20             sessions_paid      BIGINT
21               sessions_ai      BIGINT
22                ai_chatgpt      BIGINT
23             a

In [18]:
con.sql("SELECT * FROM march LIMIT 5").df()

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,<NA>,20,0,67,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,<NA>,1,0,0,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,<NA>,125,1,616,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,True,False,True,<NA>,7,0,28,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,True,False,True,<NA>,11,0,25,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.